# 90_pipelines / 91__full_pipeline

**Entry point for the Databricks Job.**  
Runs the full ingestion → transformation → analysis pipeline in sequence.

```
favorites.json              editar favoritos (00_config/favorites.json en el repo)
concept_hierarchy.json      editar jerarquía de conceptos (00_config/ en el repo)
metrics_hierarchy.json      editar jerarquía de derived metrics (00_config/ en el repo)
      ↓
02__tickers_master              build main.config.tickers
      ↓
03__concept_hierarchy_master    build main.config.concept_hierarchy
      ↓
04__metrics_hierarchy_master    build main.config.metrics_hierarchy
      ↓
11__fetch_sec_xbrl              fetch from SEC API → financials_raw
      ↓
12__fetch_market_data           fetch from Yahoo Finance → market_data
      ↓
21__clean_and_merge             deduplicate → MERGE into financials
      ↓
22__derived_metrics             margins, FCF, YoY, leverage, valuation ratios
      ↓
31__company_analysis            validation queries
```

> **Nota:** `02__tickers_master`, `03__concept_hierarchy_master` y `04__metrics_hierarchy_master` viven en `00_config/`.
> En este pipeline asumimos que `main.config.tickers` ya existe — se construye
> manualmente con `02__tickers_master`. Las jerarquías (`concept_hierarchy` y `metrics_hierarchy`)
> se reconstruyen automáticamente en cada run desde sus respectivos JSONs.

## 0. Pipeline parameters

Override at runtime via Databricks Job parameters:  
`{"tickers_override": "AAPL,TSLA", "run_optimization": "false"}`

- `tickers_override`: lista de tickers separados por coma (omite `main.config.tickers`)
- `run_optimization`: `true` para correr OPTIMIZE + VACUUM al final

> El universo de tickers (`main.config.tickers`) se mantiene manualmente con
> `02__tickers_master`. Las jerarquías de conceptos y métricas
> (`main.config.concept_hierarchy` y `main.config.metrics_hierarchy`)
> sí se reconstruyen automáticamente en cada run desde sus respectivos JSONs.

In [0]:
dbutils.widgets.text("tickers_override",  "",      "tickers_override")
dbutils.widgets.text("run_optimization",  "false", "run_optimization")
dbutils.widgets.text("rebuild_config",    "false", "rebuild_config")

tickers_override = dbutils.widgets.get("tickers_override")
run_optimization = dbutils.widgets.get("run_optimization")
rebuild_config   = dbutils.widgets.get("rebuild_config")

## 1. Load config

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/00_config/01__tickers"

In [0]:
from datetime import datetime

if tickers_override:
    ACTIVE_TICKERS = [t.strip() for t in tickers_override.split(",") if t.strip()]
    print(f"✓ Override mode — {len(ACTIVE_TICKERS)} tickers: {ACTIVE_TICKERS}")
else:
    tickers_df = spark.table(f"{CATALOG}.config.tickers")
    ACTIVE_TICKERS = [row.ticker for row in tickers_df.select("ticker").collect()]
    print(f"✓ Config loaded — {len(ACTIVE_TICKERS)} tickers from main.config.tickers")

if not ACTIVE_TICKERS:
    raise ValueError("No tickers found — run 02__tickers_master first.")

pipeline_start = datetime.utcnow()
print(f"Pipeline started at {pipeline_start.isoformat()} UTC")

## 2. Status de config tables

`main.config.tickers` debe existir antes de ejecutar este pipeline — se construye
manualmente con `02__tickers_master` (cuando edites `favorites.json`).  
`main.config.concept_hierarchy` y `main.config.metrics_hierarchy` se reconstruyen
automáticamente en los pasos 3 y 4.

In [0]:
# El rebuild de main.config.tickers no se ejecuta automáticamente porque hace
# llamadas a Wikipedia (S&P 500) y iShares (Russell 3000) que rara vez cambian.
#
# Para refrescar el universo de tickers, ejecuta 02__tickers_master manualmente
# (por ejemplo, después de editar favorites.json en el repo).
#
# Las jerarquías (siguientes pasos) sí se reconstruyen siempre — son baratas.

if rebuild_config.lower() == "true":
    print("⚠ rebuild_config=true detectado, pero el rebuild de tickers se hace")
    print("  manualmente — ejecuta 02__tickers_master por separado.")
else:
    print("⊘ Skipping ticker rebuild — usa main.config.tickers tal como está")

## 3. Concept hierarchy
`main.config.concept_hierarchy` — jerarquía de conceptos para agrupación y orden en el dashboard.

Siempre se reconstruye (es barato — ~48 filas, <5s) para asegurar que la tabla
refleja el estado actual de `concept_hierarchy.json` en el repo.

In [0]:
print("=" * 55)
print("STEP 1 / 7 — Concept Hierarchy")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/00_config/03__concept_hierarchy_master"

## 4. Metrics hierarchy
`main.config.metrics_hierarchy` — jerarquía de derived metrics (category → subcategory)
para agrupación y orden en el dashboard.

Siempre se reconstruye (es barato — ~17 filas, <2s) para asegurar que la tabla
refleja el estado actual de `metrics_hierarchy.json` en el repo.

In [0]:
print("=" * 55)
print("STEP 2 / 7 — Metrics Hierarchy")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/00_config/04__metrics_hierarchy_master"

## 5. Ingestion — fetch from SEC EDGAR
`financials_raw` — append-only audit log

In [0]:
print("=" * 55)
print("STEP 3 / 7 — SEC Ingestion")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/10_ingestion/11__fetch_sec_xbrl"

## 6. Ingestion — fetch market data from Yahoo Finance
`market_data` — year-end prices + market cap

In [0]:
print("=" * 55)
print("STEP 4 / 7 — Market Data")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/10_ingestion/12__fetch_market_data"

## 7. Clean & merge → fact table
`financials` — deduplicated, clean long-format fact table

In [0]:
print("=" * 55)
print("STEP 5 / 7 — Clean & Merge")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/20_transformation/21__clean_and_merge"

## 8. Derived metrics
`financials_metrics` — margins, FCF, YoY growth, leverage, valuation ratios

In [0]:
print("=" * 55)
print("STEP 6 / 7 — Derived Metrics")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/20_transformation/22__derived_metrics"

## 9. Analysis
Runs analysis queries — useful for validation after pipeline runs

In [0]:
print("=" * 55)
print("STEP 7 / 7 — Analysis")
print("=" * 55)

In [0]:
%run "/Workspace/Users/al.lopez.moreira@gmail.com/fundamentals_databricks_pj/FA PJ (Basic)/30_analysis/31__company_analysis"

## 10. Pipeline summary

In [0]:
pipeline_end = datetime.utcnow()
duration     = (pipeline_end - pipeline_start).total_seconds()

print(f"\n{'='*55}")
print(f"  Pipeline completed ✓")
print(f"{'='*55}")
print(f"  Started  : {pipeline_start.isoformat()} UTC")
print(f"  Finished : {pipeline_end.isoformat()} UTC")
print(f"  Duration : {duration:.1f}s ({duration/60:.1f} min)")
print(f"  Tickers  : {len(ACTIVE_TICKERS):,}")
print()

summary_tables = [
    ("config",      "tickers"),
    ("config",      "concept_hierarchy"),
    ("config",      "metrics_hierarchy"),
    (SCHEMA,        "financials_raw"),
    (SCHEMA,        "financials"),
    (SCHEMA,        "market_data"),
    (SCHEMA,        "financials_metrics"),
]

for schema, tbl in summary_tables:
    full = f"{CATALOG}.{schema}.{tbl}"
    try:
        n = spark.table(full).count()
        print(f"  {full}: {n:,} rows")
    except Exception:
        print(f"  {full}: (not found)")

## 11. Optional: optimize Delta tables
Set `run_optimization=true` in Job params to enable. Run at most once a week.

In [0]:
if run_optimization.lower() == "true":
    print("Running OPTIMIZE + VACUUM...")
    for tbl in ["financials", "market_data", "financials_metrics"]:
        full = f"{CATALOG}.{SCHEMA}.{tbl}"
        spark.sql(f"OPTIMIZE {full}")
        spark.sql(f"VACUUM   {full} RETAIN 168 HOURS")
        print(f"  ✓ {full}")
    print("Done.")
else:
    print("⊘ Skipping OPTIMIZE/VACUUM (set run_optimization=true to enable)")